# IPL T20 Cricket Graph

Load an IPL cricket graph with **Teams**, **Players**, and **Games**.

Run each cell top to bottom.

In [ ]:
import pandas as pd, numpy as np, requests, os

CP = 'http://graph-olap-control-plane:8080'
GCS = 'http://fake-gcs-local:4443'
BUCKET = 'graph-olap-local-dev'
OWNER = 'demo@example.com'
HEADERS = {'Content-Type': 'application/json', 'X-Username': OWNER, 'X-User-Role': 'admin'}

requests.post(f'{GCS}/storage/v1/b', json={'name': BUCKET})
print('Connected!')

## 1. Generate Data

In [ ]:
np.random.seed(42)

teams = pd.DataFrame({
    'team_id': range(1, 9),
    'name': ['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers', 'Kolkata Knight Riders',
             'Delhi Capitals', 'Rajasthan Royals', 'Sunrisers Hyderabad', 'Punjab Kings'],
    'city': ['Mumbai', 'Chennai', 'Bangalore', 'Kolkata', 'Delhi', 'Jaipur', 'Hyderabad', 'Mohali'],
    'titles': [5, 5, 0, 2, 0, 1, 1, 0],
})

players = pd.DataFrame({
    'player_id': range(1, 16),
    'name': ['Rohit Sharma', 'Virat Kohli', 'MS Dhoni', 'KL Rahul', 'Jasprit Bumrah',
             'Ravindra Jadeja', 'Hardik Pandya', 'Rishabh Pant', 'Shubman Gill', 'Yuzvendra Chahal',
             'Suryakumar Yadav', 'Mohammed Shami', 'Jos Buttler', 'Rashid Khan', 'David Warner'],
    'role': ['Batsman']*4 + ['Bowler', 'All-rounder', 'All-rounder', 'Wicketkeeper', 'Batsman', 'Bowler',
             'Batsman', 'Bowler', 'Batsman', 'Bowler', 'Batsman'],
})

games = pd.DataFrame({
    'match_id': range(1, 11), 'season': [2023]*5 + [2024]*5,
    'venue': ['Wankhede', 'Chepauk', 'Chinnaswamy', 'Eden Gardens', 'Kotla']*2,
    'winner_team_id': [1, 2, 4, 1, 3, 2, 5, 1, 6, 2],
})

plays_for = pd.DataFrame({'player_id': range(1, 16), 'team_id': [1,3,2,8,1,2,1,5,4,3,1,4,6,7,5]})
played_in_rows = []
for mid in range(1, 11):
    for pid in np.random.choice(range(1, 16), size=6, replace=False):
        played_in_rows.append({'player_id': int(pid), 'match_id': mid,
                               'runs': int(np.random.randint(0, 100)), 'wickets': int(np.random.randint(0, 4))})
played_in = pd.DataFrame(played_in_rows)
won = pd.DataFrame({'team_id': [1,2,4,1,3,2,5,1,6,2], 'match_id': range(1, 11)})

print(f'{len(teams)} teams, {len(players)} players, {len(games)} games')
print(f'{len(plays_for)} plays_for, {len(played_in)} played_in, {len(won)} won')
teams

## 2. Create Mapping & Instance

In [ ]:
# Create mapping
mapping = requests.post(f'{CP}/api/mappings', headers=HEADERS, json={
    'name': 'IPL T20 Cricket',
    'description': '[via:jupyter] IPL cricket graph',
    'node_definitions': [
        {'label': 'Team', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'team_id', 'type': 'INT64'}, 'properties': [{'name': 'name', 'type': 'STRING'}, {'name': 'city', 'type': 'STRING'}, {'name': 'titles', 'type': 'INT64'}]},
        {'label': 'Player', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'player_id', 'type': 'INT64'}, 'properties': [{'name': 'name', 'type': 'STRING'}, {'name': 'role', 'type': 'STRING'}]},
        {'label': 'Game', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'match_id', 'type': 'INT64'}, 'properties': [{'name': 'season', 'type': 'INT64'}, {'name': 'venue', 'type': 'STRING'}, {'name': 'winner_team_id', 'type': 'INT64'}]},
    ],
    'edge_definitions': [
        {'type': 'PLAYS_FOR', 'from_node': 'Player', 'to_node': 'Team', 'sql': 'SELECT * FROM placeholder', 'from_key': 'player_id', 'to_key': 'team_id', 'properties': []},
        {'type': 'PLAYED_IN', 'from_node': 'Player', 'to_node': 'Game', 'sql': 'SELECT * FROM placeholder', 'from_key': 'player_id', 'to_key': 'match_id', 'properties': [{'name': 'runs', 'type': 'INT64'}, {'name': 'wickets', 'type': 'INT64'}]},
        {'type': 'WON', 'from_node': 'Team', 'to_node': 'Game', 'sql': 'SELECT * FROM placeholder', 'from_key': 'team_id', 'to_key': 'match_id', 'properties': []},
    ]
}).json()
mapping_id = mapping['data']['id']
print(f'Mapping: {mapping_id}')

# Create instance
inst = requests.post(f'{CP}/api/instances', headers=HEADERS, json={
    'mapping_id': mapping_id, 'name': 'IPL T20 Demo',
    'description': '[via:jupyter]', 'wrapper_type': 'falkordb', 'ttl': 'PT4H',
}).json()
instance_id = inst['data']['id']
snapshot_id = inst['data']['snapshot_id']
print(f'Instance: {instance_id}, Snapshot: {snapshot_id}')

In [ ]:
# Save parquet and upload to GCS
import time
base = '/tmp/ipl'
for d in ['nodes/Team','nodes/Player','nodes/Game','edges/PLAYS_FOR','edges/PLAYED_IN','edges/WON']:
    os.makedirs(f'{base}/{d}', exist_ok=True)

teams.to_parquet(f'{base}/nodes/Team/data.parquet', index=False)
players.to_parquet(f'{base}/nodes/Player/data.parquet', index=False)
games.to_parquet(f'{base}/nodes/Game/data.parquet', index=False)
plays_for.to_parquet(f'{base}/edges/PLAYS_FOR/data.parquet', index=False)
played_in.to_parquet(f'{base}/edges/PLAYED_IN/data.parquet', index=False)
won.to_parquet(f'{base}/edges/WON/data.parquet', index=False)

prefix = f'{OWNER}/{mapping_id}/v1/{snapshot_id}'
for name in ['nodes/Team','nodes/Player','nodes/Game','edges/PLAYS_FOR','edges/PLAYED_IN','edges/WON']:
    with open(f'{base}/{name}/data.parquet', 'rb') as f:
        requests.post(f'{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={prefix}/{name}/data.parquet',
                      data=f.read(), headers={'Content-Type': 'application/octet-stream'})
print('Data uploaded!')

# Wait for instance
for i in range(60):
    status = requests.get(f'{CP}/api/instances/{instance_id}', headers=HEADERS).json()['data']['status']
    if status == 'running': break
    time.sleep(5)

pod_name = requests.get(f'{CP}/api/instances/{instance_id}', headers=HEADERS).json()['data']['pod_name']
W = f'http://{pod_name}:8000'
print(f'Instance running at {W}')

## 3. Query

In [ ]:
def q(cypher):
    r = requests.post(f'{W}/query', json={'query': cypher})
    d = r.json()
    return pd.DataFrame(d['rows'], columns=d['columns'])

q('MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count ORDER BY count DESC')

In [ ]:
# Which team won the most?
q('MATCH (t:Team)-[:WON]->(g:Game) RETURN t.name AS team, count(g) AS wins ORDER BY wins DESC')

In [ ]:
# Top scorers
q('''
MATCH (p:Player)-[pi:PLAYED_IN]->(g:Game)
RETURN p.name AS player, sum(pi.runs) AS total_runs, sum(pi.wickets) AS total_wickets
ORDER BY total_runs DESC LIMIT 10
''')

In [ ]:
# Team rosters
q('MATCH (p:Player)-[:PLAYS_FOR]->(t:Team) RETURN t.name AS team, collect(p.name) AS players ORDER BY team')

In [ ]:
# Players in most games
q('''
MATCH (p:Player)-[:PLAYED_IN]->(g:Game)
RETURN p.name AS player, p.role AS role, count(g) AS games
ORDER BY games DESC LIMIT 10
''')

## 4. Visualize Graph

In [ ]:
from graph_olap_sdk import visualize

visualize(W, limit=200, title='IPL T20 Cricket')

## Cleanup

In [ ]:
# Uncomment to terminate:
# requests.delete(f'{CP}/api/instances/{instance_id}', headers=HEADERS)
# print('Done!')